# ============================================================
#   TailorTrack — Order & Measurement Management System
#   Course  : MGT-173 Introduction to Programming
# ============================================================

 ── File names ──

In [ ]:
import os
# file name constants are defined once here so we don't have to retype them
CUSTOMERS_FILE = "customers.txt"
ORDERS_FILE    = "orders.txt"
PAYMENTS_FILE  = "payments.txt"

CLASS: FileManager

In [ ]:

class FileManager:     #Handles all file operations (saving and loading data)

    def save(self, filename, data):  #Saves a list of strings to a file, one per line
       try:
            file = open(filename, "w")
            for line in data:
                file.write(line + "\n")
            file.close()
            
       except Exception as e:       # if disk is full or permissions are wrong, we want to know
            print("  Error: Could not save file.", e)


    def load(self, filename):            #Loads a list of strings from a file, one per line
        if not os.path.exists(filename):   # if file does NOT exist, return empty list
            return []
        try:                                # EXCEPTION HANDLING: protect against corrupted files
            file  = open(filename, "r")
            lines = file.read().splitlines()  # read whole file and split into list
            file.close()
            return [line for line in lines if line.strip() != ""]  # filter empty lines
        except Exception as e:           # if file is corrupted or unreadable
            print("  Error: Could not read file.", e)
            return []  


def get_numeric_input(prompt, value_type=float, allow_empty=False, empty_value=None): #function to get numeric input 
    while True:  # loop until we get valid input
        try:
            value = input(prompt)   
            if allow_empty and value == "":   # if empty input is allowed and user entered empty string, return the specified empty value
                return empty_value
            return value_type(value)        
        except ValueError:
            print("  Error: Please enter a valid number. Try again.")
         

Class: Customer

In [ ]:
class Customer:    #Handles customer-related operations (add, view, search, update)

    def __init__(self):        #Constructor initializes the FileManager(object) instance for this class to use
        self.fm = FileManager()
        

    def next_id(self):                     #Method to generate next customer ID
        data = self.fm.load(CUSTOMERS_FILE)
        if not data:
            return "C001"                  # if there are no customers yet, start with C001
        max_num = 0
        for line in data:                  
            parts = line.split("|")        # split the line into parts using the pipe separator
            try:
                num = int(parts[0][1:])   # strip the "C" prefix to get the number
                if num > max_num:
                    max_num = num
            except ValueError:
                pass
        return f"C{max_num + 1:03d}"       # f-string formats it as 3 digits
    

    def _line_to_dict(self, line):          # helper method to convert a pipe-separated line into a dictionary for easier access
        parts = line.split("|")
        return {
            "customer_id" : parts[0],
            "name"        : parts[1],
            "contact"     : parts[2],
            "chest"       : parts[3],
            "waist"       : parts[4],
            "length"      : parts[5]
        }
        

    def add_customer(self):                    #method to collect customer info and save it to file
        print("\n--- Add New Customer ---")
        name    = input("  Name           : ")   
        contact = input("  Contact Number : ").strip()
        chest   = get_numeric_input("  Chest in Inches   : ", float)
        waist   = get_numeric_input("  Waist in Inches   : ", float)
        length  = get_numeric_input("  Length in Inches  : ", float)

       
        if name.strip() == "" or contact == "":        
            print("  Name and contact cannot be empty.")
            return

        cid = self.next_id()                     #generate next unique ID

    
        customer = {                             #dictionary to store customer information
            "customer_id" : cid,
            "name"        : name.strip(),
            "contact"     : contact,       # string, not int -- preserves leading zero
            "chest"       : str(chest),
            "waist"       : str(waist),
            "length"      : str(length)
        }

       
        line = "|".join(customer.values())   # convert the dictionary values into a pipe-separated string for storage

        data = self.fm.load(CUSTOMERS_FILE)
        data.append(line)
        self.fm.save(CUSTOMERS_FILE, data)
        print("  Customer added! ID:", cid)
        

    def view_all(self):                       #Display all customers using dictionary access
        print("\n--- All Customers ---")
        data = self.fm.load(CUSTOMERS_FILE)
        if not data:
            print("  No customers found.")
            return
        for line in data:          
            c = self._line_to_dict(line)                  
            print(f"  ID: {c['customer_id']} | Name: {c['name']} | Contact: {c['contact']}")


    def search(self):                                    #Search for customers by name or ID
        print("\n--- Search Customer ---")
        keyword = input("  Enter name or ID: ").lower()
        data    = self.fm.load(CUSTOMERS_FILE)
        found   = False                                  ## BOOLEAN FLAG: tracks if any match found
        for line in data:
            c = self._line_to_dict(line)
            if keyword in c["name"].lower() or keyword == c["customer_id"].lower():
                print(f"  ID: {c['customer_id']} | Name: {c['name']} | Contact: {c['contact']}")
                print(f"  Measurements — Chest: {c['chest']} | Waist: {c['waist']} | Length: {c['length']}")
                found = True
        if not found:
            print("  No customer found.")
            

    def update_measurements(self):
        print("\n--- Update Measurements ---")
        cid  = input("  Enter Customer ID: ").upper()
        data = self.fm.load(CUSTOMERS_FILE)
        
        for i in range(len(data)):                              # FOR LOOP with index — needed to modify list items directly
            c = self._line_to_dict(data[i])
            if c["customer_id"] == cid:
                print("  Press Enter to keep current value.")    # pass the current value as default so pressing Enter skips the field
        
                chest  = get_numeric_input(f"  Chest  [{c['chest']}]: ", float, True, c["chest"])
                waist  = get_numeric_input(f"  Waist  [{c['waist']}]: ", float, True, c["waist"])
                length = get_numeric_input(f"  Length [{c['length']}]: ", float, True, c["length"])     # update the dict then write it back as a pipe-separated line
                
                c["chest"]  = str(chest)
                c["waist"]  = str(waist)
                c["length"] = str(length)
                data[i] = "|".join(c.values())
                self.fm.save(CUSTOMERS_FILE, data)
                print("  Measurements updated!")
                return
        print("  Customer not found.")
    

    def delete_customer(self):              # METHOD: Remove a customer record permanently from file
        print("\n--- Delete Customer ---")
        cid  = input("  Enter Customer ID to delete: ").upper()
        data = self.fm.load(CUSTOMERS_FILE)

        updated = [line for line in data if self._line_to_dict(line)["customer_id"] != cid] # create a new list excluding the customer to delete

        if len(updated) == len(data):
            print("  Customer not found.")
            return

        confirm = input("  Are you sure? This cannot be undone (yes/no): ").lower()
        if confirm != "yes":
            print("  Delete cancelled.")
            return

        self.fm.save(CUSTOMERS_FILE, updated)
        print("  Customer deleted successfully.")
   

Class: Order

In [ ]:
class Order:

    def __init__(self):         # Constructor — sets up FileManager when Order() is created
        self.fm = FileManager()

    def next_id(self):             # Method to generate next order ID
        data = self.fm.load(ORDERS_FILE)     
        if not data:                
            return "O001"
        max_num = 0
        for line in data:             # loop through existing orders to find the highest order number
            parts = line.split("|")   
            try:
                num = int(parts[0][1:])  # strip the "O" prefix to get the number
                if num > max_num:
                    max_num = num
            except ValueError:
                pass 
        return f"O{max_num + 1:03d}"    
    

    def _line_to_dict(self, line):       # helper method to convert a pipe-separated line into a dictionary for easier access
        parts = line.split("|")
        return {                          # creates a dictionary with keys corresponding to order attributes and values 
            "order_id"      : parts[0],
            "customer_id"   : parts[1],
            "dress_type"    : parts[2],
            "delivery_date" : parts[3],
            "status"        : parts[4],
            "advance"       : parts[5]
        }
        

    def create_order(self):                    
        print("\n--- Create New Order ---")        
        cid      = input("  Customer ID          : ").upper()  #uppercase
        dress    = input("  Dress Type           : ")
        delivery = input("  Delivery Date DD/MM/YY: ")         
        advance  = get_numeric_input("  Advance (Rs)         : ", float, True, 0) # allow empty input for advance, treat as 0

        if cid == "" or dress == "" or delivery == "":     
            print("  All fields are required.")
            return

        oid = self.next_id()              #generate next unique order ID

       
        order = {                         #dictionary to store order information
            "order_id"      : oid,
            "customer_id"   : cid,
            "dress_type"    : dress,
            "delivery_date" : delivery,
            "status"        : "Pending",
            "advance"       : str(advance)
        }

        data = self.fm.load(ORDERS_FILE)         # load existing orders, append the new one, and save back to file
        data.append("|".join(order.values()))
        self.fm.save(ORDERS_FILE, data)
        print("  Order created! ID:", oid)
        

    def view_all(self):                    
        print("\n--- All Orders ---")        
        data = self.fm.load(ORDERS_FILE)
        if not data:
            print("  No orders found.")
            return
        for line in data:                   # loop through each line, convert to dict for easy access, and print summary info
            o = self._line_to_dict(line)
            print(f"  ID: {o['order_id']} | Customer: {o['customer_id']} | Dress: {o['dress_type']}"  
                  f" | Delivery: {o['delivery_date']} | Status: {o['status']}")    
            

    def update_status(self):                 
        print("\n--- Update Order Status ---") 
        oid      = input("  Enter Order ID: ").upper()
        statuses = ["Pending", "In Progress", "Ready", "Delivered"]
        print("  1. Pending  2. In Progress  3. Ready  4. Delivered")
        data = self.fm.load(ORDERS_FILE)
        
        for i in range(len(data)):            # FOR LOOP with index — needed to modify list items directly 
            o = self._line_to_dict(data[i])
            if o["order_id"] == oid:
                choice = input("  Select (1-4): ")
                if choice in ["1", "2", "3", "4"]:           #Conditional check to ensure valid input before updating status
                    o["status"] = statuses[int(choice) - 1]
                    data[i]     = "|".join(o.values())
                    self.fm.save(ORDERS_FILE, data)
                    print("  Status updated to:", o["status"])
                else:
                    print("  Invalid choice.")
                return
        print("  Order not found.")
        

    def track_order(self):
        print("\n--- Track Order ---")    
        oid  = input("  Enter Order ID: ").upper()
        data = self.fm.load(ORDERS_FILE)
        
        for line in data:                    # loop through orders to find the one with matching ID and print detailed info
            o = self._line_to_dict(line)
            if o["order_id"] == oid:          
                print(f"  Order ID     : {o['order_id']}")
                print(f"  Customer ID  : {o['customer_id']}")
                print(f"  Dress Type   : {o['dress_type']}")
                print(f"  Delivery Date: {o['delivery_date']}")
                print(f"  Status       : {o['status']}")
                print(f"  Advance Paid : Rs {o['advance']}")
                return
        print("  Order not found.")
        

    def orders_by_customer(self):               # method to display all orders for a specific customer ID
        print("\n--- Orders by Customer ---")
        cid   = input("  Enter Customer ID: ").upper()
        data  = self.fm.load(ORDERS_FILE)
        found = False
        for line in data:
            o = self._line_to_dict(line)           # loop through orders line by line
            if o["customer_id"] == cid:
                print(f"  ID: {o['order_id']} | Dress: {o['dress_type']}"
                      f" | Delivery: {o['delivery_date']} | Status: {o['status']}")
                found = True                       
        if not found:
            print("  No orders found for this customer.")
            

    def pending_orders(self):                         # method to display all orders that are still pending or in progress
        print("\n--- Pending / In-Progress Orders ---")
        data  = self.fm.load(ORDERS_FILE)
        found = False                           # boolean flag to track if we found any pending orders
        for line in data:
            o = self._line_to_dict(line)
            if o["status"] == "Pending" or o["status"] == "In Progress":
                print(f"  ID: {o['order_id']} | Customer: {o['customer_id']} | Dress: {o['dress_type']}"
                      f" | Due: {o['delivery_date']} | Status: {o['status']}")
                found = True
        if not found:
            print("  No pending orders.")
            
    
    def delete_order(self):
        print("\n--- Delete Order ---")
        oid  = input("  Enter Order ID to delete: ").upper()
        data = self.fm.load(ORDERS_FILE)  # load all orders into list

        updated = [line for line in data if self._line_to_dict(line)["order_id"] != oid]

        if len(updated) == len(data):
            print("  Order not found.")
            return

        confirm = input("  Are you sure? This cannot be undone (yes/no): ").lower()
        if confirm != "yes":
            print("  Delete cancelled.")
            return

        self.fm.save(ORDERS_FILE, updated)  
        print("  Order deleted successfully.")
   

Class: Payment


In [ ]:
class Payment:           #Handles payment-related operations (add payment, view payment, unpaid orders, revenue summary)            

    def __init__(self):          # Constructor — sets up FileManager when Payment() is created
        self.fm = FileManager()

    def next_id(self):                    # Method to generate next payment ID
        data = self.fm.load(PAYMENTS_FILE)
        if not data:
            return "P001"
        max_num = 0                    # variable to track the highest payment number found so far                    
        for line in data:                
            parts = line.split("|")
            try:
                num = int(parts[0][1:])   # strip the "P" prefix
                if num > max_num:         # update max_num if this payment's number is higher 
                    max_num = num
            except ValueError:
                pass
        return f"P{max_num + 1:03d}"

    def _line_to_dict(self, line):           # helper method to convert a pipe-separated line into a dictionary for easier access
        """Parse a stored payment line back into a dictionary."""
        parts = line.split("|")
        return {
            "payment_id" : parts[0],
            "order_id"   : parts[1],
            "total"      : parts[2],
            "paid"       : parts[3],
            "remaining"  : parts[4],
            "status"     : parts[5]
        }

    def add_payment(self):                     
        print("\n--- Add Payment ---")
        oid   = input("  Order ID        : ").upper()
        total = get_numeric_input("  Total Bill (Rs) : ", float) # get_numeric_input is a helper function that handles input validation and conversion to the specified type (float in this case)
        paid  = get_numeric_input("  Amount Paid (Rs): ", float)

        remaining = total - paid 
        status    = "Paid" if remaining <= 0 else "Unpaid"
        if remaining < 0:     # if they paid more than the total, we consider it fully paid and ignore the negative remaining amount
            remaining = 0   

        pid = self.next_id() 
        payment = {
            "payment_id" : pid,
            "order_id"   : oid,
            "total"      : str(total),
            "paid"       : str(paid),
            "remaining"  : str(remaining),
            "status"     : status
        }

        data = self.fm.load(PAYMENTS_FILE)   # load existing payments, append the new one, and save back to file
        data.append("|".join(payment.values()))
        self.fm.save(PAYMENTS_FILE, data)

        print("  Payment recorded!")            # print a summary of the payment details
        print(f"  Total     : Rs {total}")
        print(f"  Paid      : Rs {paid}")
        print(f"  Remaining : Rs {remaining}")
        print(f"  Status    : {status}")

    def view_payment(self):                     # method to view payment details for a specific order ID
        print("\n--- View Payment ---")
        oid  = input("  Enter Order ID: ").upper()
        data = self.fm.load(PAYMENTS_FILE)
        for line in data:
            pay = self._line_to_dict(line)
            if pay["order_id"] == oid:
                print(f"  Payment ID  : {pay['payment_id']}")
                print(f"  Order ID    : {pay['order_id']}")
                print(f"  Total Bill  : Rs {pay['total']}")
                print(f"  Amount Paid : Rs {pay['paid']}")
                print(f"  Remaining   : Rs {pay['remaining']}")
                print(f"  Status      : {pay['status']}")
                return
        print("  No payment found for this order.")
 
    def unpaid_orders(self):                # method to display all orders that have an unpaid balance
        print("\n--- Unpaid Orders ---")
        data  = self.fm.load(PAYMENTS_FILE)
        found = False                        # boolean flag to track if we found any unpaid orders
        for line in data:                    
            pay = self._line_to_dict(line)
            if pay["status"] == "Unpaid":
                print(f"  Order: {pay['order_id']} | Bill: Rs {pay['total']}"
                      f" | Paid: Rs {pay['paid']} | Remaining: Rs {pay['remaining']}")
                found = True
        if not found:
            print("  All orders are fully paid!")

    def total_revenue(self):                # method to calculate and display total revenue (billed, received, pending)
        print("\n--- Revenue Summary ---")   # print header for the revenue summary section
        data = self.fm.load(PAYMENTS_FILE)
        if not data:                        
            print("  No payment records.")
            return
        billed   = 0
        received = 0
        pending  = 0
        for line in data:
            pay       = self._line_to_dict(line)
            billed   += float(pay["total"])
            received += float(pay["paid"])
            pending  += float(pay["remaining"])
        print(f"  Total Billed   : Rs {billed}")
        print(f"  Total Received : Rs {received}")
        print(f"  Total Pending  : Rs {pending}")
        
 
    def delete_payment(self):
        print("\n--- Delete Payment ---")
        pid = input("  Enter Payment ID to delete: ").upper()
        data = self.fm.load(PAYMENTS_FILE)  # load all payments into list

        updated = [line for line in data if self._line_to_dict(line)["payment_id"] != pid]

        if len(updated) == len(data):
            print("  Payment not found.")
            return

        confirm = input("  Are you sure? This cannot be undone (yes/no): ").lower()
        if confirm != "yes":
            print("  Delete cancelled.")
            return

        self.fm.save(PAYMENTS_FILE, updated)
        print("  Payment deleted successfully.")

Menu Functions

In [ ]:
def customer_menu(c):                         # Function Displays the customer management menu and handles user choices
    while True:                               # WHILE LOOP, keeps menu showing after every action
        print("\n===== CUSTOMER MENU =====")
        print("  1. Add Customer")
        print("  2. View All")
        print("  3. Search Customer")
        print("  4. Update Measurements")
        print("  5. Delete Customer")
        print("  0. Back")
        print("==========================")
        choice = input("  Select: ")           ## USER INPUT: stored as string e.g. "1", "2", "0"
        if   choice == "1": c.add_customer()    ## OOP, calling method on object
        elif choice == "2": c.view_all()
        elif choice == "3": c.search()
        elif choice == "4": c.update_measurements()
        elif choice == "5": c.delete_customer()
        elif choice == "0": break
        else: print("  Invalid option.")


def order_menu(o):
    while True:
        print("\n===== ORDER MENU =====")
        print("  1. Create Order")
        print("  2. View All Orders")
        print("  3. Update Status")
        print("  4. Track Order")
        print("  5. Orders by Customer")
        print("  6. Pending Orders")
        print("  7. Delete Order")
        print("  0. Back")
        print("=======================")
        choice = input("  Select: ")
        if   choice == "1": o.create_order()
        elif choice == "2": o.view_all()
        elif choice == "3": o.update_status()
        elif choice == "4": o.track_order()
        elif choice == "5": o.orders_by_customer()
        elif choice == "6": o.pending_orders()
        elif choice == "7": o.delete_order()
        elif choice == "0": break
        else: print("  Invalid option.")


def payment_menu(p):
    while True:
        print("\n===== PAYMENT MENU =====")
        print("  1. Add Payment")
        print("  2. View Payment")
        print("  3. Unpaid Orders")
        print("  4. Revenue Summary")
        print("  5. Delete Payment")
        print("  0. Back")
        print("=========================")
        choice = input("  Select: ")
        if   choice == "1": p.add_payment()
        elif choice == "2": p.view_payment()
        elif choice == "3": p.unpaid_orders()
        elif choice == "4": p.total_revenue()
        elif choice == "5": p.delete_payment()
        elif choice == "0": break
        else: print("  Invalid option.")

Main

In [19]:
def main():    # Creating objects (instances) of each class to use in the main menu
    c = Customer()
    o = Order()
    p = Payment()

    print("\n========================================")
    print("       Welcome to TailorTrack \U0001f9f5        ")
    print("  Order & Measurement Management System ")
    print("========================================")

    while True:                            # Infinite loop to keep the main menu running
        print("\n===== MAIN MENU =====")
        print("  1. Customer Management")
        print("  2. Order Management")
        print("  3. Payment Management")
        print("  0. Exit")
        print("======================")
        choice = input("  Select: ")
        if   choice == "1": customer_menu(c)
        elif choice == "2": order_menu(o)
        elif choice == "3": payment_menu(p)
        elif choice == "0":
            print("\n  Goodbye! \U0001f9f5\n") # print a goodbye message with a tailor emoji before exiting
            break
        else:
            print("  Invalid option.")

if __name__ == "__main__":                 # This line checks whether this file is being run directly (as the main program) rather than imported as a module in another file. If this condition is true, it calls the main() function to start the program.
    main()


  Goodbye! 🧵

